# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ridoy1211/Flyrank-Internship-ML/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

Paper: *The State of AI-Driven SEO* (FlyRank, March 2026) — 341,701 content pieces, 57 brands, headline findings from direct aggregate comparisons, with an exploratory ML appendix on 61,790 active content records.

---

### Finding 1 — "What Predicts Health?" (Random Forest feature importance)

**The finding, in my own words:** the paper runs a holdout-tested Random Forest to find which features best predict a page's Health Score, and reports Average Position (43%), Impressions (32%), and Scroll Depth (15%) as the top predictors.

**Where the label comes from:** a defined rule, not an observed outcome — Health Score is explicitly a FlyRank composite: Impressions (30 pts) + Position (30 pts) + CTR (20 pts) + Scroll Depth (20 pts). The paper itself discloses this and calls the result "descriptive rather than causal," which is exactly the right instinct.

**Does the validation design carry the claim?** Not fully, and the paper's own caveat doesn't go quite far enough. Holdout-testing protects against a model overfitting *noise* in the training split — it does nothing to protect against a model finding a formula's own inputs inside itself, which is guaranteed to happen regardless of how the data is split. This is the exact label-derived-feature pattern from the `hunting-leakage-and-validating` skill: the label was computed FROM these columns, so of course they dominate the importance ranking. A 90/10 holdout split versus a 50/50 one would change nothing here — the circularity isn't a sample-size problem, it's a design problem the split can't fix.

**My question:** given that Position and Impressions are literally 60 of the 100 points in the Health Score formula, what would this feature-importance chart look like against an outcome that *isn't* partly built from the same inputs — e.g., future 30-day impression or click growth? That would tell readers something the current version structurally cannot: whether these signals predict something new, rather than predicting a formula that already contains them.

---

### Finding 2 — "What Predicts Growth?" (Logistic Regression, 71% holdout accuracy)

**The finding, in my own words:** a logistic regression, described as 71% holdout accuracy, separates "growing" from "declining" pages, reporting Content Age as the strongest negative signal and Days Visible / recent impressions as the strongest positive signals.

**Where the label comes from:** a defined rule (rising vs. falling impressions over a window) — the same shape as the `decoupling_signature` proxy I've built since Week 2, and worth naming explicitly as a proxy, not an observed business outcome like revenue or retained traffic.

**Does the validation design carry the claim?** This is the one I'd push on hardest. The paper's own earlier aggregate finding (Finding #1 in the paper, a different section) reports 74,800 growing pages vs. 45,600 declining pages in a similar up/down split — a base rate of roughly 62% positive. The ML appendix never states the base rate for its own 61,790-row sample used for this specific classifier. Per the `hunting-leakage-and-validating` skill's own example almost exactly: *"accuracy of 71% on a label that is 62% positive is 9 points of skill, not 71."* If the ML sample's true split is close to that same ~62%, this model's real lift over just guessing the majority class is roughly 9 points — a real but much more modest result than "71% accuracy" reads on its own.

**My question:** what was the actual positive rate in the 61,790-row holdout sample used for this specific model, and how does 71% accuracy compare to that number? Reporting accuracy next to the base rate costs one extra sentence and would let a reader immediately judge real lift instead of being impressed by a raw percentage that could mean anything from strong skill to barely-better-than-guessing.

---

Both questions are asked in the same spirit the paper asks of itself elsewhere — it already discloses "no p-values or confidence intervals are reported" and treats its own ML pages as secondary to direct aggregate evidence. These two questions extend that same self-critical standard one step further, the way I'd want my Week 3-5 work read.

In [ ]:
# Fill in once the paper is available — no query needed for this section.
pass


## 2. My model under an honest split (before/after)

Week 5 already used a grouped split (`GroupShuffleSplit` on `client_hash_id`) rather than a plain random one — so the honest "before/after" comparison here is exactly the gap the `hunting-leakage-and-validating` skill asks for: **random split vs. grouped split, same data, same model, same metric.** If the random-split number is meaningfully higher, that gap itself is memorization, not skill.

Rebuilding the Week 3–5 pipeline self-contained below (Feb = features, March = label).

In [ ]:
%pip -q install duckdb huggingface_hub scikit-learn

import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb, numpy as np, pandas as pd
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {'dim_content': f"read_parquet('{REL}/dim_content.parquet')"}
FEB = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-02/*.parquet')"
MAR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

feb_page = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS impressions_prior30,
           SUM(gsc_clicks)      AS clicks_prior30,
           CASE WHEN SUM(gsc_impressions) > 0
                THEN SUM(gsc_sum_position) / SUM(gsc_impressions) ELSE NULL END AS avg_position_prior30
    FROM {FEB} WHERE gsc_data_available IS TRUE GROUP BY 1, 2 HAVING SUM(gsc_impressions) > 0
""").df()
feb_page['ctr_prior30'] = feb_page['clicks_prior30'] / feb_page['impressions_prior30'] * 100

content_meta = con.sql(f"SELECT content_hash_id, content_type, main_intent FROM {TABLES['dim_content']}").df()
feb_page = feb_page.merge(content_meta, on='content_hash_id', how='left')

mar_page = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS impressions_last30, SUM(gsc_clicks) AS clicks_last30
    FROM {MAR} WHERE gsc_data_available IS TRUE GROUP BY 1, 2
""").df()

df = feb_page.merge(mar_page, on=['client_hash_id', 'content_hash_id'], how='inner')
df['impr_change_pct'] = 100 * (df['impressions_last30'] - df['impressions_prior30']) / df['impressions_prior30']
df['click_change_pct'] = 100 * (df['clicks_last30'] - df['clicks_prior30']) / df['clicks_prior30'].replace(0, np.nan)
df['decoupling_signature'] = (df['impr_change_pct'].between(-10, 10) & (df['click_change_pct'] <= -15)).astype(int)
df = df.dropna(subset=['avg_position_prior30', 'content_type', 'main_intent'])

print(f'{len(df):,} rows, positive rate: {df["decoupling_signature"].mean():.4f}')


In [ ]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

feature_cols_numeric = ['impressions_prior30', 'clicks_prior30', 'ctr_prior30', 'avg_position_prior30']
feature_cols_categorical = ['content_type', 'main_intent']
all_features = feature_cols_numeric + feature_cols_categorical

preprocess = ColumnTransformer([
    ('num', StandardScaler(), feature_cols_numeric),
    ('cat', OneHotEncoder(handle_unknown='ignore'), feature_cols_categorical),
])

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

def fit_and_score(train_df, test_df, label):
    X_train, y_train = train_df[all_features], train_df[label]
    X_test, y_test = test_df[all_features], test_df[label]
    model = Pipeline([('prep', preprocess), ('clf', RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1))])
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    return {
        'precision@50': precision_at_k(proba, y_test.values, 50),
        'roc_auc': roc_auc_score(y_test, proba),
        'base_rate': y_test.mean(),
        'n_test': len(test_df),
    }

# BEFORE: plain random split (the dishonest default)
random_train, random_test = train_test_split(df, test_size=0.25, random_state=42, stratify=df['decoupling_signature'])
random_result = fit_and_score(random_train, random_test, 'decoupling_signature')

# AFTER: grouped split by client (the honest split, same as Week 5)
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['client_hash_id']))
grouped_train, grouped_test = df.iloc[train_idx], df.iloc[test_idx]
overlap = set(grouped_train['client_hash_id']) & set(grouped_test['client_hash_id'])
grouped_result = fit_and_score(grouped_train, grouped_test, 'decoupling_signature')

before_after = pd.DataFrame([
    {'split': 'Random (BEFORE — dishonest)', **random_result},
    {'split': 'Grouped by client (AFTER — honest)', **grouped_result},
])
before_after['gap_vs_honest'] = before_after['precision@50'] - grouped_result['precision@50']

print(f'Clients leaking across train/test in the grouped split: {len(overlap)} (should be 0)')
before_after


**Reading the before/after:** *(fill in with your real numbers once run)* if the random-split precision@50 sits noticeably above the grouped one, that gap is the model quietly memorizing client-specific patterns rather than learning something that generalizes to a client it's never seen — the exact failure mode a random split hides and a grouped split exposes. If the two numbers are close, that's also worth reporting plainly: it suggests client identity wasn't doing much hidden work in this particular feature set.

## 3. Leakage audit

Running the `hunting-leakage-and-validating` attack checklist against the final Week 5 feature set, not just asserting it's clean.

In [ ]:
# 1) Timeline check: every feature column name should reference the PRIOR window only
suspect_terms = ['last30', 'decoupling_signature', 'trend_direction', 'trend_pct', 'march', 'mar_']
flagged = [c for c in all_features if any(term in c.lower() for term in suspect_terms)]
print('1) Features referencing the outcome window or the label itself:', flagged or 'none')
assert not flagged, 'Leakage: a future-window or label-derived column is in the feature list.'

# 2) No product-flag / existing-system-score features (none exist in this data by design, but check anyway)
product_flag_terms = ['health_score', 'priority_score', 'action_type', 'baseline_score', 'refresh_tier']
flagged_flags = [c for c in all_features if any(term in c.lower() for term in product_flag_terms)]
print('2) Product-flag-style features:', flagged_flags or 'none')
assert not flagged_flags

# 3) Population check: does row inclusion depend on outcome-window info?
print('3) Row inclusion rule: content item must have usable data in BOTH Feb and March '
      '(inner join). This is itself a population choice that depends on March having any rows '
      'at all for that page — disclosed here, not hidden: a page that churned entirely by March '
      '(zero rows) is silently excluded rather than counted as a strong negative.')

# 4) Split grouped by repeating entity — already shown in Section 2 (0 overlapping clients)
print('4) Grouped split check: done in Section 2, 0 overlapping clients confirmed.')

# 5) Base rate printed next to metrics — already done in Section 2's before_after table
print('5) Base rate: included in the before/after table above for both splits.')


In [ ]:
# 6) Deliberate re-add of a known-leaky column (same trap as Week 3), to prove the harness catches it
leak_check_df = df.copy()
leak_features = all_features + ['clicks_last30']

leak_train, leak_test = train_test_split(leak_check_df, test_size=0.25, random_state=42, stratify=leak_check_df['decoupling_signature'])

preprocess_leak = ColumnTransformer([
    ('num', StandardScaler(), feature_cols_numeric + ['clicks_last30']),
    ('cat', OneHotEncoder(handle_unknown='ignore'), feature_cols_categorical),
])
leak_model = Pipeline([('prep', preprocess_leak), ('clf', RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1))])
leak_model.fit(leak_train[feature_cols_numeric + ['clicks_last30'] + feature_cols_categorical], leak_train['decoupling_signature'])
leak_proba = leak_model.predict_proba(leak_test[feature_cols_numeric + ['clicks_last30'] + feature_cols_categorical])[:, 1]
leak_auc = roc_auc_score(leak_test['decoupling_signature'], leak_proba)

print(f'Honest ROC AUC (final feature set):        {grouped_result["roc_auc"]:.3f}')
print(f'Deliberately-leaked ROC AUC (+clicks_last30): {leak_auc:.3f}')
print('If the leaked number jumps well above the honest one, the test harness correctly '
      'detects leakage when it is actually present — confirming a clean score elsewhere is '
      'trustworthy, not just untested.')


## 4. Claim rewrite

**My boldest sentence from Week 4/5 work, as originally written:**
> *"A page scores above zero only if it already has real visibility and its February CTR already sits below what's typical for pages at its own position — exactly the leading pattern behind Great Decoupling."*

**Rewritten in safe, public-safe language:**
> *We observed that pages with below-median February click-through rate for their position bucket were directionally more likely to show the impressions-flat/clicks-down signature in March, in this data slice. This is a decision-support signal for prioritizing manual review, not a claim that a CTR gap causes or predicts decoupling for any individual page, and it has not been tested against seasonality or SERP-layout explanations.*

**What changed and why:** the original sentence states a mechanism ("the leading pattern behind Great Decoupling") as if it were established causally. The rewrite: (1) uses "observed"/"directionally" instead of asserting a mechanism, (2) names the specific slice the claim is scoped to, (3) explicitly disclaims causation, and (4) names a specific untested alternative explanation (seasonality/SERP changes) instead of implying none exist.

In [ ]:
# Claim rewrite is text-only; no query needed for this section.
pass


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.